## Download the exercise data
Run the next cell once before starting the exercise. It downloads and extracts this notebook’s data into `~/kenya2026`. Set `KENYA2026_WORK_DIR` first if you prefer another location.

In [ ]:
from pathlib import Path
import os
import subprocess

exercise = "day5_morning_heterozygosity_roh"
base_url = "https://popgen.dk/albrecht/course/kenya2026/data"
work_dir = Path(os.environ.get("KENYA2026_WORK_DIR", Path.home() / "kenya2026")).expanduser()
exercise_dir = work_dir / exercise
archive = work_dir / f"{exercise}.zip"
work_dir.mkdir(parents=True, exist_ok=True)

if not archive.exists():
    subprocess.run(["wget", "-c", f"{base_url}/{exercise}.zip", "-O", str(archive)], check=True)
if not exercise_dir.exists():
    subprocess.run(["unzip", "-q", str(archive), "-d", str(work_dir)], check=True)

os.chdir(exercise_dir)
print(f"Working directory: {Path.cwd()}")

### Software requirements
The setup cell above downloads **only the exercise data**. It does not install software. Before running the rest of this notebook, install the command-line programs and the Python or R packages that are imported or called in the exercises. If you see an error such as `command not found`, `ModuleNotFoundError`, or `there is no package called ...`, install the named dependency or ask an instructor for help.

# Genetic diversity practical: heterozygosity and ROH

A statistic commonly used to represent the genetic diversity within an individual is the proportion of sites that have two different alleles, referred to as heterozygosity. While heterozygosity can refer to the proportion out of any set of markers, in this exercise we will look at the number of heterozygous sites out of all callable sites. Heterozygosity, while being a measure for the within-individual genetic variation, also provides information about the genetic variation in the population as a whole.

In this exercise we will cover:
 - Filtering data to get accurate calls of heterozygous sites
 - How to estimate heterozygosity for a single individual
 - Why the denominator matters
 - Comparing heterozygosity among individuals, populations, species, and genomic windows
 - Calling and interpreting runs of homozygosity for two example individuals

Tools used: bcftools, samtools, AWK, Python/jupyterquiz, R, plink

The notebooks are editable, so feel free to experiment and change the code to see what happens, or to write notes in the text cells.

First, we define the paths for the files we need during the exercise.

In [ ]:
### data paths
BAM=CTauTzS_8872.Goat.bam
GOODSITES=Goat.siteQC.good.bed
GOAT_REF=goat.fa.gz
HET=het.roh.tsv
CHR27_BCF=CTauTzS_8872.chr27.filtered.bcf.gz
HET_TRAG=heterozygosity_trag.txt

### make sure required software is installed
which bcftools
which samtools
which awk

### make directory for the exercise
mkdir -p "$HOME/kenya2026/GeneticDiversity"
echo "Exercise output directory: $HOME/kenya2026/GeneticDiversity"

Do you remember from earlier what a bam file contains? Let's try to look inside the BAM file we're going to be using as input by running the following:

In [ ]:
samtools view CTauTzS_8872.Goat.bam | head -n1

This outputs the information for a single read coming from a single wildebeest individual, the one with ID number 8872. The BAM file contains around 242 million such reads - only from one individual! Pause a bit and think about the enormity of this data, and how much information it contains. That's the power of whole-genome sequencing.

## Filtering
Since we are using data that has not been filtered, and because heterozygosity estimation depends a lot on correctly determining which sites are truly heterozygous, we will want to employ some filters on the mapped reads. But first we will need to know the average sequencing depth of our data. This is done by first extracting the depth for every site mapped to goat chromosome 27 or "NC_030834.1" with "samtools depth" and then piping this result into AWK to compute the mean.

In [ ]:
### compute depth
samtools depth CTauTzS_8872.Goat.bam -r NC_030834.1 |  awk '{sum+=$3} END { print "Mean = ",sum/NR}'

Two of the filters we are going to use are given as options to bcftools below: `-Q 30 -q 25`, which tell the program to ignore bases in reads if their base calling quality score is below 30 and to ignore entire reads if their mapping quality score is below 25.
 - What is the difference between base calling quality score and mapping quality score?
 - What do values of 30 and 25 correspond to? (Hint: https://en.wikipedia.org/wiki/Phred_quality_score)

In addition to the quality filters, we will also set a minimum and maximum depth of sequencing for each site. Here, we exclude all sites from the estimation if they have less than half of the mean sequencing depth we just calculated, or if they have more than twice the mean depth. We add this filter because sites with unusually low or high depth are likely to be problematic. For example, high depth can arise due to paralogy, when two very similar regions in the sampled species map to a single region in the reference genome.

In addition to these basic filters, there are several others that can be considered before calling genome-wide heterozygosity. For example, repetitive sequence regions are spread across the genome of many organisms. The goat genome has about 45% repetitive sequence. Such regions can make mapping difficult and can make genotype calls unreliable. One way to identify these regions is with RepeatMasker. We will not go into the details here, but we have prepared a list of sites in advance that passed more extensive filtering. This list of "good sites" can be supplied to bcftools with the `-T` option.

It looks like this inside:

In [ ]:
head Goat.siteQC.good.bed

Run the code below to start a short quiz about filtering and callable sites.

In [ ]:
from jupyterquiz import display_quiz
display_quiz('/davidData/users/thomas/workshop/heterozygosity_quiz1_filtering.json')

Now we are almost ready to apply these filters and call genotypes on sites that pass the filters. However, going through an entire genome and evaluating every single position takes time and processing power, and so for the purpose of demonstration we will be using data mapped to only one small region, namely positions 30,000,000 to 31,000,000 on goat chromosome  27. In a real study, we would want to base the heterozygosity estimation on as much data as possible to get the most accurate estimate. 
 - How large a portion of chromosome 27 is this subset of 1 million bases? What about compared to the whole genome? (Hint: Look here https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_001704415.2/)
 
We now apply our filters and call genotypes in the small region:

In [ ]:
### filtering
set -euo pipefail

bcftools mpileup --threads 10 --full-BAQ -r NC_030834.1:30000000-31000000 -T Goat.siteQC.good.bed -Q 30 -q 25 -O u \
    --fasta-ref goat.fa.gz --per-sample-mF -a FORMAT/AD,FORMAT/DP CTauTzS_8872.Goat.bam | \
    bcftools call -Ob -o "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.bcf.gz" --threads 10 -c

Next, we want to do some more processing of the output and fill out the tag "AC", which is short for "allele count". The command may look a bit scary, but if we break it down it should not be too bad. In the first part, the expression after -i removes sites where the reference or alternate allele is not a single base (non-SNPs), as well as sites where the number of reads covering it is less than half or more than double the mean depth of coverage. The next part removes any site that is called heterozygous but where only a single read supports one of the two alleles, because this is unlikely if the site is truly heterozygous, but can happen because of a single erroneous base call. The last part fills in the tag "AC" for the number of alternate alleles in the remaining sites, which is useful for inspecting the file. For the actual heterozygosity count below, we count directly from the genotype field (`GT`). 

In [ ]:
### more filtering and filling AC tag
set -euo pipefail

bcftools view --threads 10 -i 'strlen(REF)==1 & (strlen(ALT)==1 || ALT=".") &  FMT/DP>=8 & FMT/DP<=34' \
    -M 2 -Ou "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.bcf.gz" | \
    bcftools view --threads 10 -i '(GT=="het" & FMT/AD[*:0]>=2 & FMT/AD[*:1]>=2 ) || GT=="hom"' | \
    bcftools +fill-tags /dev/stdin -Ob -o "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.filtered.bcf.gz" -- -t AC

Let's have a look at the resulting file (-H skips the header lines that contain metadata):

In [ ]:
bcftools view -H "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.filtered.bcf.gz" | head -n500 | column -t 

 Note that lines above are too long to fit in the box, so they will wrap around to the next line, so only every other line is actually a new line.
 - Try to see if you can find a heterozygous site in the cell above. (Hint: Start by looking at sites where the "ALT" field is not "." and then look at the genotype or "GT")
 - If a site is truly heterozygous, what proportion of reads covering that site would you expect to support each of the two alleles?

## Estimation
Above, we made a cautiously filtered BCF file containing genotypes from a small region of chromosome 27. From here, the actual heterozygosity estimation is quite simple in concept: count the heterozygous sites and divide by the number of callable sites.

The important detail is the denominator. We are not dividing by the whole genome length, and we are not dividing only by variable SNPs. We are dividing by the sites that survived our filters and were therefore considered callable for this individual in this analysis.

We can count genotype classes by querying the `GT` field and using AWK to count how often each genotype occurs.

In [ ]:
# Count genotype classes in the filtered subset we just made.
set -euo pipefail

bcftools query -f '[%GT]\n' "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.filtered.bcf.gz" | \
awk '
  $1 ~ /^0[\/|]0$/ {hom_ref++}
  $1 ~ /^0[\/|]1$|^1[\/|]0$/ {het++}
  $1 ~ /^1[\/|]1$/ {hom_alt++}
  END {
    total = hom_ref + het + hom_alt
    if (total == 0) {
      print "ERROR: no callable genotypes were counted. Check that CTauTzS_8872.subset.filtered.bcf.gz exists and contains GT calls." > "/dev/stderr"
      exit 1
    }
    print ".", hom_ref + 0
    print "1", het + 0
    print "2", hom_alt + 0
  }
' > "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.AC"

In [ ]:
cat "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.AC"

Saved to the file above we have the counts of sites that are homozygous for the reference allele, sites that are heterozygous, and sites that are homozygous for the alternative allele.

- `.` means homozygous reference genotype (`0/0`).
- `1` means heterozygous genotype (`0/1` or `1/0`).
- `2` means homozygous alternate genotype (`1/1`).

What do you think could be the reason for the large number of sites homozygous for the alternative allele? Hint: what is the reference genome, and what species is this individual?

Run the code below to start a short quiz about the denominator in heterozygosity estimates.

In [ ]:
from jupyterquiz import display_quiz
display_quiz('/davidData/users/thomas/workshop/heterozygosity_quiz2_denominator.json')

In [ ]:
# Heterozygosity = heterozygous sites / all callable sites.
awk '
  $1=="." {hom_ref=$2}
  $1=="1" {het=$2}
  $1=="2" {hom_alt=$2}
  END {
    total = hom_ref + het + hom_alt
    print "hom_ref_sites =", hom_ref
    print "het_sites     =", het
    print "hom_alt_sites =", hom_alt
    print "callable_sites=", total
    print "heterozygosity=", het / total
  }
' "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.AC"

### What if we only used variable sites?

A common trap is to estimate heterozygosity only from sites already known to be variable. That changes the denominator. Below we compare the heterozygosity estimate using all callable sites with a SNP-only style estimate that ignores homozygous reference sites.

In [ ]:
awk '
  $1=="." {hom_ref=$2}
  $1=="1" {het=$2}
  $1=="2" {hom_alt=$2}
  END {
    callable_total = hom_ref + het + hom_alt
    alt_site_total = het + hom_alt
    print "all callable sites:       ", het / callable_total
    print "ALT/SNP sites only:       ", het / alt_site_total
    print "fold difference:          ", (het / alt_site_total) / (het / callable_total)
  }
' "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.subset.AC"

- Why is the SNP-only value so different?
- Which estimate of the two do you think is closer to the "true" heterozygosity?
- When might a SNP-only estimate still be useful?

In [ ]:
# Now count genotype classes for the precomputed whole chromosome 27 file.
set -euo pipefail

bcftools query -f '[%GT]\n' CTauTzS_8872.chr27.filtered.bcf.gz | \
awk '
  $1 ~ /^0[\/|]0$/ {hom_ref++}
  $1 ~ /^0[\/|]1$|^1[\/|]0$/ {het++}
  $1 ~ /^1[\/|]1$/ {hom_alt++}
  END {
    total = hom_ref + het + hom_alt
    if (total == 0) {
      print "ERROR: no callable genotypes were counted. Check that CHR27_BCF exists and contains GT calls." > "/dev/stderr"
      exit 1
    }
    print ".", hom_ref + 0
    print "1", het + 0
    print "2", hom_alt + 0
  }
' > "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.chr27.AC"

cat "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.chr27.AC"

awk '
  $1=="." {hom_ref=$2}
  $1=="1" {het=$2}
  $1=="2" {hom_alt=$2}
  END {
    total = hom_ref + het + hom_alt
    print "chromosome_27_heterozygosity =", het / total
  }
' "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.chr27.AC"

Now compare the subset estimate to a precomputed file covering all of chromosome 27. This is the same filtering idea as above, but applied to much more sequence.

- Why do you get different values on the whole chromosome compared to the small region/subset?
- Which estimate should we usually trust more, and why?
- Then try to run the filtering and estimation again with one or more filters changed or removed. For example, look at `-T`, `-Q`, `-q`, or the depth cutoffs. What changes?

## Heterozygosity Along a Chromosome

A single chromosome-wide value can still hide local variation. Here we estimate heterozygosity in 1 Mb windows along chromosome 27 using the precomputed filtered BCF file. This is still heterozygosity, just summarized locally instead of across the whole chromosome.

In [ ]:
set -euo pipefail

bcftools query -f '%POS[\t%GT]\n' CTauTzS_8872.chr27.filtered.bcf.gz | \
awk -v w=1000000 '
  BEGIN {OFS="\t"; print "window_start", "window_end", "callable_sites", "het_sites", "heterozygosity"}
  {
    bin = int(($1 - 1) / w) + 1
    total[bin]++
    if ($2 ~ /^0[\/|]1$|^1[\/|]0$/) het[bin]++
  }
  END {
    for (b = 1; b <= 60; b++) {
      if (total[b] > 0) print (b - 1) * w + 1, b * w, total[b], het[b] + 0, (het[b] + 0) / total[b]
    }
  }
' > "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.chr27.window_het.tsv"

head -n 20 "$HOME/kenya2026/GeneticDiversity/CTauTzS_8872.chr27.window_het.tsv" | column -t

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
window_het <- read.table(path.expand("~/kenya2026/GeneticDiversity/CTauTzS_8872.chr27.window_het.tsv"), header = TRUE)

# Simple binomial SE for the proportion of heterozygous sites.
# This ignores linkage, genotype uncertainty, and mapping/callability effects.
window_het$se <- with(window_het,
    sqrt(heterozygosity * (1 - heterozygosity) / callable_sites))

with(window_het, {
    x <- window_start / 1e6
    plot(x, heterozygosity, type="b", pch=19, col="#006f68",
         xlab="Chromosome 27 position (Mb)", ylab="Heterozygosity")
    arrows(x, heterozygosity-se, x, heterozygosity+se,
           angle=90, code=3, length=0.03, col="#006f68")
})

abline(h=mean(window_het$heterozygosity), col="#d96c4f", lty=2)

- Are heterozygous sites evenly distributed along the chromosome?
- What could cause windows with unusually low heterozygosity?
- Which possibilities are biological, and which are technical?

This windowed view is a natural bridge to ROH: long stretches with very low heterozygosity may reflect recent shared ancestry, but low callability, mapping problems, or other filters can also produce local dips.

## Comparison between individuals
In isolation the heterozygosity value we just estimated does not tell us much, so we will need something to compare it to. In general, compared to other large African mammals, this value is on the low side, but let's look at some more wildebeest samples in comparison.
While we could have estimated values for more individuals by repeating the previous procedure, we have cheated a bit and done this ahead of time, saving the heterozygosities estimated on whole genomes in a file where the relevant values look like this:

In [ ]:
cut het.roh.tsv -f1,6,7 | column -t

To get a better overview of these values, we can plot them in, for example, a boxplot separated by a grouping of interest such as population, locality, or species. Here the field "map" denotes locality, and we can use this information to separate the samples into groups.

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 10)
het_table <- read.table("het.roh.tsv", header = TRUE)
boxplot(het ~ map, data = het_table, col = "#4fb3a4", ylab = "Heterozygosity")

Now we have a better overview of the distributions of heterozygosity.
- What could be a reason that we see differences in heterozygosity between some populations, but not so much within the different populations?
- What could be the reason for the outliers we see? (the dots at Etosha and Monduli)
- The value we got for individual 8872 was lower than the other wildebeest populations. Individual 8872 comes from Nyerere National Park in Tanzania (formerly Selous Game Reserve). Can you suggest an explanation of why it has lower heterozygosity?
  
Finally, we have also estimated heterozygosity in a similar manner from a range of other species belonging to the Tragelaphines, or spiral-horned antelopes. Here we plot these values to allow comparison with the wildebeests.

In [ ]:
het_table_trag <- read.table("heterozygosity_trag.txt", header = FALSE)
names <- c("Tory" = "eland", "Tder" = "giant_eland", "Tstr" = "greater_kudu", "Timb" = "lesser_kudu", "Tbux" = "mountain_nyala", "Scaf" = "nyala", "Tspe" = "sitatunga", "Tscr" = "bushbuck")

het_table_trag$species <- names[substr(het_table_trag$V1, 1, 4)]
het_table_wildebeest <- data.frame(V1 = het_table$sampleID, V2 = het_table$het, species = rep("wildebeest", dim(het_table)[1]))
het_table_species <- rbind(het_table_trag, het_table_wildebeest)

boxplot(V2 ~ species, data = het_table_species, col = "#d96c4f", ylab = "Heterozygosity")

- What extra technical caveats matter when comparing heterozygosity across species mapped to the same goat reference genome?

# Part 2: Runs of Homozygosity

Genome-wide heterozygosity gives one summary value per individual, but it does not tell us how homozygosity is arranged along the genome. Runs of homozygosity, or ROH, are long stretches where an individual is homozygous at many consecutive SNPs. Long ROH are usually interpreted as segments inherited identical-by-descent from a recent shared ancestor.

Here we compare two wildebeest individuals with contrasting ROH patterns:

- `CTauBwC__761`
- `CTauZmE_2542`

The PLINK file we use here is close to the starting point used in the original article, but it is not quite the final ROH input. The original project also masked heterozygous genotypes with suspicious allele balance before PLINK conversion, but we skip that step here for technical reasons. We still use the rest of the filtering as in the article: population-specific MAF and missingness filters, followed by removal of sites with unusually high observed heterozygosity.


In [ ]:
### Work inside the course directory.
mkdir -p "$HOME/kenya2026/GeneticDiversity/ROH"
echo "ROH output directory: $HOME/kenya2026/GeneticDiversity/ROH"

### Check that the files and tools exist.
ls -lh Wildebeest_wildebeest_variable_sites_nomultiallelics_noindels_10dp_3het.MAF005.bed \
       Wildebeest_wildebeest_variable_sites_nomultiallelics_noindels_10dp_3het.MAF005.bim \
       Wildebeest_wildebeest_variable_sites_nomultiallelics_noindels_10dp_3het.MAF005.fam
/emily/program/bin/plink --version


We define population lists that should be confirmed with analysis of population structure. For this exercise we derive two small groups from sample-name prefixes so the filtering is transparent and self-contained. In a real analysis, we would use the curated population assignment file from the project.

The `--allow-extra-chr` option is needed because the wildebeest assembly uses scaffold names such as `HiC_scaffold_1` rather than chromosome codes like `1`, `2`, `3`.


In [ ]:
### Population-like groups for the two target individuals.
ROHDIR="$HOME/kenya2026/GeneticDiversity/ROH"
mkdir -p "$ROHDIR"

awk '$2 ~ /^CTauBwC/ {print $1, $2}' Wildebeest_wildebeest_variable_sites_nomultiallelics_noindels_10dp_3het.MAF005.fam > "$ROHDIR/CTauBwC.list"
awk '$2 ~ /^CTauZmE/ {print $1, $2}' Wildebeest_wildebeest_variable_sites_nomultiallelics_noindels_10dp_3het.MAF005.fam > "$ROHDIR/CTauZmE.list"

wc -l "$ROHDIR/CTauBwC.list" "$ROHDIR/CTauZmE.list"
head "$ROHDIR/CTauBwC.list" "$ROHDIR/CTauZmE.list"


First apply filters at the group level:

- `--maf 0.05`: keep variants that are polymorphic enough within the group
- `--geno 0.05`: remove variants with too much missing data
- `--hardy`: calculate observed heterozygosity per SNP

Then remove sites where the observed heterozygosity is 0.5 or above. These high-heterozygosity sites are suspicious for ROH calling because they can reflect mapping/paralogy problems and can break true homozygous tracts into smaller pieces.

This is a useful contrast with the heterozygosity part of the exercise. For genome-wide heterozygosity we cared a lot about defining callable bases. For ROH we need a marker set that does not contain many false heterozygous genotypes.


In [ ]:
set -euo pipefail
cd "$HOME/kenya2026/GeneticDiversity/ROH"

for pop in CTauBwC CTauZmE; do
  echo "### Filtering $pop"

  /emily/program/bin/plink \
    --bfile Wildebeest_wildebeest_variable_sites_nomultiallelics_noindels_10dp_3het.MAF005 \
    --keep ${pop}.list \
    --allow-extra-chr \
    --maf 0.05 \
    --geno 0.05 \
    --hardy \
    --make-bed \
    --out ${pop}_maf05_geno005

  awk 'NR > 1 && $7 >= 0.5 {print $2}' ${pop}_maf05_geno005.hwe > ${pop}_OHE05.sites

  ### This text report is large and not needed after extracting the high-het site list.
  rm -f ${pop}_maf05_geno005.hwe

  echo -n "Variants after MAF/missingness filtering: "
  wc -l ${pop}_maf05_geno005.bim
  echo -n "High-observed-heterozygosity sites to exclude: "
  wc -l ${pop}_OHE05.sites
  echo
 done


Before moving on:

- Why do we remove SNPs with very high observed heterozygosity before calling ROH?
- Why are the MAF and missingness filters applied within the two population-like groups rather than across all wildebeest samples at once?


Now we infer ROH for the two focal individuals with the following parameters:

- `--homozyg-kb 1000`: only report ROH at least 1 Mb long
- `--homozyg-window-het 3`: allow a few heterozygous calls inside the sliding window
- `--homozyg-window-missing 20`: allow some missing calls inside the sliding window

Allowing a few heterozygous or missing calls is practical because genotype errors and missing data can otherwise break true ROH into smaller pieces.


In [ ]:
set -euo pipefail
cd "$HOME/kenya2026/GeneticDiversity/ROH"

for sample in CTauBwC__761 CTauZmE_2542; do
  if [ "$sample" = "CTauBwC__761" ]; then
    pop=CTauBwC
  else
    pop=CTauZmE
  fi

  printf '0 %s\n' "$sample" > ${sample}.keep

  /emily/program/bin/plink \
    --bfile ${pop}_maf05_geno005 \
    --keep ${sample}.keep \
    --exclude ${pop}_OHE05.sites \
    --allow-extra-chr \
    --homozyg \
    --homozyg-kb 1000 \
    --homozyg-window-het 3 \
    --homozyg-window-missing 20 \
    --out ${sample}_OHE05

  ### PLINK can write a very large per-SNP summary file. We do not need it here.
  rm -f ${sample}_OHE05.hom.summary
 done


Before looking at the plots:

- What would a false heterozygous genotype do to a true homozygous tract?
- Why do we allow a few heterozygous or missing calls inside the ROH window instead of requiring perfect homozygosity at every SNP?


The individual summary gives the number of ROH segments (`NSEG`), total length in ROH (`KB`), and average ROH length (`KBAVG`).


In [ ]:
cd "$HOME/kenya2026/GeneticDiversity/ROH"

for sample in CTauBwC__761 CTauZmE_2542; do
  echo "### $sample"
  column -t ${sample}_OHE05.hom.indiv
  echo
 done


Removing suspicious heterozygous sites can increase the number or length of inferred ROH, because false heterozygous genotypes can break long homozygous tracts.

Look at the longest segments for each individual.


In [ ]:
cd "$HOME/kenya2026/GeneticDiversity/ROH"

for sample in CTauBwC__761 CTauZmE_2542; do
  echo "### Longest ROH for $sample"
  sort -k9,9nr ${sample}_OHE05.hom | head -10 | column -t
  echo
 done


For plotting, we make group-level filtered PLINK files and then thin them to every 20th SNP. The ROH calls above are based on the denser filtered marker set; thinning is only for making the example plotting command fast enough to run.

The plot shows:

- blue: retained SNP/scaffold track
- red: heterozygous SNPs
- purple: SNP density
- grey: heterozygosity in windows
- black/grey horizontal segments: PLINK ROH

The schematic below labels these tracks in the output of the plotting script:

![Annotated ROH plot showing heterozygous sites, homozygous sites, inferred ROH, window heterozygosity, and SNP density](https://raw.githubusercontent.com/popgenDK/courses/refs/heads/main/kenya2024/exercises/day2/ROHplot.png)

Because the input is a variable-site SNP dataset, the red marks are heterozygous SNPs among retained SNPs. This is not the same visual scale as heterozygous bases divided by all callable bases.


In [ ]:
set -euo pipefail
cd "$HOME/kenya2026/GeneticDiversity/ROH"

for pop in CTauBwC CTauZmE; do
  /emily/program/bin/plink \
    --bfile ${pop}_maf05_geno005 \
    --exclude ${pop}_OHE05.sites \
    --allow-extra-chr \
    --make-bed \
    --out ${pop}_OHE05_group

  awk 'NR % 20 == 0 {print $2}' ${pop}_OHE05_group.bim > ${pop}_OHE05_group_plot.extract

  /emily/program/bin/plink \
    --bfile ${pop}_OHE05_group \
    --allow-extra-chr \
    --extract ${pop}_OHE05_group_plot.extract \
    --make-bed \
    --out ${pop}_OHE05_group_plot
 done


In [ ]:
set -euo pipefail
cd "$HOME/kenya2026/GeneticDiversity/ROH"

Rscript /davidData/users/thomas/workshop/plotPlinkROH.R \
  -p CTauBwC_OHE05_group_plot.bed \
  -s CTauBwC__761 \
  --homfile CTauBwC__761_OHE05.hom

Rscript /davidData/users/thomas/workshop/plotPlinkROH.R \
  -p CTauZmE_OHE05_group_plot.bed \
  -s CTauZmE_2542 \
  --homfile CTauZmE_2542_OHE05.hom


The PNGs displayed below were precomputed from the full, unthinned filtered marker sets, so they show the complete retained SNP tracks rather than the faster thinned plotting version generated above.

In [ ]:
from IPython.display import Image, display

roh_plot_dir = '/davidData/users/thomas/workshop/roh_full_plots'

display(Image(filename=f'{roh_plot_dir}/CTauBwC__761.ROH.Density.png'))
display(Image(filename=f'{roh_plot_dir}/CTauZmE_2542.ROH.Density.png'))


Important: You can inspect the plots better by right-clicking them and pressing "open image in new tab"

Questions:

- Which individual has more long ROH?
- Are the long ROH concentrated on a few scaffolds or spread across the genome?
- Why might heterozygosity outside long ROH be useful when comparing individuals from different demographic histories?


In [ ]:
from jupyterquiz import display_quiz

display_quiz('/davidData/users/thomas/workshop/roh_quiz1_filtering.json')


## ROH Length Classes Across Individuals

We can summarize ROH as a fraction of the genome in different length classes. The total/sum of these is called `F_ROH`: total length in ROH divided by the genome length considered.

Computing this for many individuals from the full PLINK dataset would take longer than we want in the exercise, so here we use precomputed ROH length-class summaries:

- ROH from 1-2 Mb
- ROH from 2-5 Mb
- ROH from 5-10 Mb
- ROH longer than 10 Mb

The point here is interpretation: both the total amount of ROH and the length distribution matter.

We also include the Etosha population here, which the article reports to have experienced ancient black-wildebeest introgression in the past.


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)

het_table <- read.table("het.roh.tsv", header = TRUE)

selected_maps <- c("B-Etosha", "C-Luangwa", "E.wBearded_KeSNai", "W.wBearded")
pop_labels <- c("B-Etosha" = "Etosha",
                "C-Luangwa" = "Luangwa",
                "E.wBearded_KeSNai" = "Nairobi",
                "W.wBearded" = "Serengeti")

roh_long <- het_table %>%
  filter(map %in% selected_maps) %>%
  mutate(
    map = factor(map, levels = selected_maps),
    total_froh = X1 + X2 + X3 + X4
  ) %>%
  arrange(map, total_froh) %>%
  mutate(sampleID = factor(sampleID, levels = sampleID)) %>%
  select(sampleID, map, total_froh, X1, X2, X3, X4) %>%
  pivot_longer(c(X1, X2, X3, X4), names_to = "roh_class", values_to = "froh") %>%
  mutate(roh_class = recode(roh_class,
                            X1 = "1-2 Mb",
                            X2 = "2-5 Mb",
                            X3 = "5-10 Mb",
                            X4 = ">10 Mb"),
         roh_class = factor(roh_class, levels = c("1-2 Mb", "2-5 Mb", "5-10 Mb", ">10 Mb")))

ggplot(roh_long, aes(x = sampleID, y = froh, fill = roh_class)) +
  geom_col(width = 0.85, color = "grey30", linewidth = 0.12) +
  facet_grid(. ~ map, scales = "free_x", space = "free_x",
             labeller = as_labeller(pop_labels)) +
  scale_fill_manual(values = c("1-2 Mb" = "#b8e0d6",
                               "2-5 Mb" = "#72b7a6",
                               "5-10 Mb" = "#2f887f",
                               ">10 Mb" = "#104f55")) +
  labs(x = NULL, y = expression(F[ROH]), fill = "ROH length") +
  theme_classic(base_size = 12) +
  theme(
    legend.position = "top",
    strip.background = element_rect(fill = "grey92", color = NA),
    strip.text = element_text(face = "bold"),
    axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6),
    panel.spacing.x = unit(0.7, "lines"),
    plot.margin = margin(8, 8, 8, 8)
  )

roh_long %>%
  group_by(map, sampleID) %>%
  summarise(total_froh = sum(froh), .groups = "drop") %>%
  group_by(map) %>%
  summarise(n = n(),
            mean_froh = mean(total_froh),
            min_froh = min(total_froh),
            max_froh = max(total_froh),
            .groups = "drop")


Questions:

- Which group has the highest total `F_ROH`?
- Do high-ROH individuals mostly have short ROH, long ROH, or both?
- Why can very long ROH be interpreted as evidence of more recent inbreeding than short ROH?
- Does the population with the highest heterozygosity also have the highest `F_ROH`? What does that tell us?


## Comparing heterozygosity with and without ROH

The table used earlier contains genome-wide heterozygosity (`het`) and heterozygosity after masking long ROH (`hetNoROH`). Here each individual gets one stacked bar:

- the darker base is genome-wide heterozygosity
- the lighter cap on top is the extra heterozygosity recovered after masking/removing long ROH
- the full bar height is heterozygosity outside long ROH


In [ ]:
library(ggplot2)
library(dplyr)

het_table <- read.table("het.roh.tsv", header = TRUE)

selected_maps <- c("B-Etosha", "C-Luangwa", "E.wBearded_KeSNai", "W.wBearded")
pop_labels <- c("B-Etosha" = "Etosha",
                "C-Luangwa" = "Luangwa",
                "E.wBearded_KeSNai" = "Nairobi",
                "W.wBearded" = "Serengeti")
pop_cols <- c("B-Etosha" = "#8a6f2a",
              "C-Luangwa" = "#1f6f78",
              "E.wBearded_KeSNai" = "#2f7d4f",
              "W.wBearded" = "#6f5aa7")

plot_table <- het_table %>%
  filter(map %in% selected_maps) %>%
  mutate(
    map = factor(map, levels = selected_maps),
    roh_remainder = pmax(hetNoROH - het, 0)
  ) %>%
  arrange(map, hetNoROH) %>%
  mutate(sampleID = factor(sampleID, levels = sampleID))

plot_long <- bind_rows(
  plot_table %>%
    transmute(sampleID, map,
              component = "Genome-wide het",
              heterozygosity = het),
  plot_table %>%
    transmute(sampleID, map,
              component = "Remainder up to het outside long ROH",
              heterozygosity = roh_remainder)
) %>%
  mutate(component = factor(component,
                            levels = c("Remainder up to het outside long ROH",
                                       "Genome-wide het")))

ggplot(plot_long, aes(x = sampleID, y = heterozygosity,
                      fill = map, alpha = component)) +
  geom_col(width = 0.85, color = "grey30", linewidth = 0.12) +
  facet_grid(. ~ map, scales = "free_x", space = "free_x",
             labeller = as_labeller(pop_labels)) +
  scale_fill_manual(values = pop_cols) +
  scale_alpha_manual(
    values = c("Genome-wide het" = 1,
               "Remainder up to het outside long ROH" = 0.35),
    breaks = c("Genome-wide het",
               "Remainder up to het outside long ROH"),
    labels = c("Genome-wide het",
               "Extra het after masking long ROH")
  ) +
  guides(fill = "none",
         alpha = guide_legend(title = NULL,
                              override.aes = list(fill = "#287f7a",
                                                  color = "grey25"))) +
  labs(x = NULL, y = "Heterozygosity") +
  theme_classic(base_size = 12) +
  theme(
    legend.position = "top",
    strip.background = element_rect(fill = "grey92", color = NA),
    strip.text = element_text(face = "bold"),
    axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6),
    panel.spacing.x = unit(0.7, "lines"),
    plot.margin = margin(8, 8, 8, 8)
  )

plot_table %>%
  group_by(map) %>%
  summarise(n = n(),
            mean_het = mean(het),
            mean_hetNoROH = mean(hetNoROH),
            mean_remainder = mean(roh_remainder),
            .groups = "drop")


Questions:

- Which individuals show the largest gain in heterozygosity?
- What would it mean if one individual had low genome-wide heterozygosity, but fairly normal heterozygosity outside long ROH?
- Etosha has high heterozygosity. Based on previous exercises or the article, what historical process could help explain that?
